# Person 4 behaviour features — train-only ML experiment

Source: Thanmaya's `notebooks/04_behaviour_features.ipynb`, branch `behaviour-features-pr-04`, commit `193568616947812e5d2e7d1db1180b06c3827752`.

The six feature calculations, payment valuation and reference checks are preserved. This local integration generates **train only** to evaluate the contribution without reading official test data. Outputs are isolated under `outputs/thanmaya_behaviour/`. No labels are used in feature computation. ML uses our existing seed-42 split, rather than the seed-0 suggestion in the source handoff text. No feature definitions or time-window boundaries were changed.

## 1. Environment and file locations

Use Python 3.12. The tested packages are numpy 2.3.1, pandas 2.3.1, pyarrow 24.0.0 and ipykernel 7.4.0. If needed, install them once in your environment using `python -m pip install numpy==2.3.1 pandas==2.3.1 pyarrow==24.0.0 ipykernel==7.4.0`. No other project Python files are imported.

Prerequisites: Person 1 has generated data/processed/train_cleaned.parquet and test_cleaned.parquet. Actual sender-paid USD is not supplied there, so we also read the corresponding raw transfers.parquet and market_rates.csv. Edit the paths below if needed.


Integration environment: the existing CPython 3.13.2 environment is reused; installed package versions are recorded in the comparison report. No package upgrade or downgrade is performed.

In [ ]:
from pathlib import Path
from collections import Counter
from dataclasses import dataclass, asdict
from decimal import Decimal
import json
import numpy as np
import pandas as pd

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/raw").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the team repository, or set ROOT to its path.")
CLEANED_DIR = ROOT / "data/processed"
RAW_DIR = ROOT / "data/raw"
OUTPUT_DIR = ROOT / "outputs/thanmaya_behaviour"
XRP = "0x1111222211112222111122221111222211112222"
print("Repository:", ROOT)


## 2. The six calculations

Windows are [t−1 hour,t) or [t−24 hours,t): the lower boundary is included and **every payment at t is excluded**. The previous gap uses the latest strictly earlier outgoing payment, even if older than 24 hours, and stays missing if none is observed. Inflow means payments received by the current sender. Unknown amounts make their affected window sum unknown; empty histories have zero flow. Inputs are copied, never changed. Sorting, binary searches and sliding counters avoid filtering the entire dataset per row.


In [ ]:
FEATURES = [
    'sender_tx_count_1h', 'sender_tx_count_24h', 'seconds_since_previous_tx',
    'sender_unique_recipients_24h', 'sender_outflow_usd_24h', 'sender_inflow_usd_24h',
]


@dataclass(frozen=True)
class Columns:
    transaction_hash: str = 'TRANSACTION_HASH'
    timestamp: str = 'TIMESTAMP'
    sender: str = 'FROM_ADDRESS'
    recipient: str = 'TO_ADDRESS'
    paid_usd: str = 'PAID_USD'
    received_usd: str = 'RECEIVED_USD'


def _validated(frame: pd.DataFrame, columns: Columns) -> pd.DataFrame:
    names = list(asdict(columns).values())
    if not frame.columns.is_unique or len(set(names)) != len(names):
        raise ValueError('Input columns and mapped field names must be unique.')
    missing = set(names) - set(frame.columns)
    if missing:
        raise ValueError(f'Missing required columns: {sorted(missing)}')
    data = frame[names].copy().reset_index(drop=True)
    data.columns = list(asdict(columns))
    for field in ['transaction_hash', 'sender', 'recipient']:
        if not data[field].map(lambda x: isinstance(x, str) and bool(x.strip())).all():
            raise ValueError(f'{field} must contain nonempty strings with no missing values.')
    if data.transaction_hash.duplicated().any():
        raise ValueError('Expected one row per TRANSACTION_HASH; duplicate hashes found.')
    # Explicit integer Unix seconds: never guess milliseconds, strings or timezone.
    try:
        times = pd.to_numeric(data.timestamp, errors='raise').to_numpy(dtype=float)
    except (ValueError, TypeError) as exc:
        raise ValueError('timestamp must be integer Unix seconds.') from exc
    if (not np.isfinite(times).all() or (times < 0).any()
            or (times > 10_000_000_000).any() or (times != np.floor(times)).any()
            or data.timestamp.map(lambda x: isinstance(x, (bool, np.bool_))).any()):
        raise ValueError('timestamp must be nonmissing integer Unix seconds, not milliseconds.')
    data['timestamp'] = times.astype(np.int64)
    for field in ['paid_usd', 'received_usd']:
        try:
            amounts = pd.to_numeric(data[field], errors='raise').to_numpy(dtype=float, na_value=np.nan)
        except (ValueError, TypeError) as exc:
            raise ValueError(f'{field} must contain numeric USD amounts or missing values.') from exc
        if np.isinf(amounts).any() or (amounts < 0).any() or data[field].map(lambda x: isinstance(x, (bool, np.bool_))).any():
            raise ValueError(f'{field} must contain nonnegative finite USD amounts or missing values.')
        data[field] = amounts
    return data


def _window_sum(times, amounts, query_times):
    """Unknown amount in a nonempty window => NaN; empty window => zero."""
    left = np.searchsorted(times, query_times - 86400, side='left')
    right = np.searchsorted(times, query_times, side='left')
    known = ~np.isnan(amounts)
    # Missing values are excluded only from arithmetic; a separate count ensures
    # they always make the resulting window unknown, never a zero-valued payment.
    values = np.zeros(len(amounts), dtype=float)
    values[known] = amounts[known]
    with np.errstate(over='raise', invalid='raise'):
        sums = np.concatenate(([0.0], np.cumsum(values)))
        result = sums[right] - sums[left]
    unknown = np.concatenate(([0], np.cumsum(~known)))
    result[unknown[right] > unknown[left]] = np.nan
    return result


def build_features(frame: pd.DataFrame, columns: Columns = Columns()) -> pd.DataFrame:
    """Use only this frame's history. Preserve row order and never mutate input.

    Windows are [t-width, t). All rows at t are excluded, irrespective of index.
    Payment principal only: fees and intermediate liquidity-provider legs must
    already be excluded from the two supplied USD fields.
    """
    data = _validated(frame, columns)
    n = len(data)
    output = pd.DataFrame({'TRANSACTION_HASH': data.transaction_hash.to_numpy(copy=True)})
    for name in FEATURES:
        output[name] = np.zeros(n, dtype=np.int64 if name in (FEATURES[0], FEATURES[1], FEATURES[3]) else float)
    output['seconds_since_previous_tx'] = np.nan
    ordered = data.sort_values('timestamp', kind='stable')
    incoming = {wallet: group for wallet, group in ordered.groupby('recipient', sort=False)}
    for wallet, group in ordered.groupby('sender', sort=False):
        indexes = group.index.to_numpy()
        times = group.timestamp.to_numpy()
        before = np.searchsorted(times, times, side='left')
        lower24 = np.searchsorted(times, times - 86400, side='left')
        output.loc[indexes, FEATURES[0]] = before - np.searchsorted(times, times - 3600, side='left')
        output.loc[indexes, FEATURES[1]] = before - lower24
        gaps = np.full(len(times), np.nan)
        observed = before > 0
        gaps[observed] = times[observed] - times[before[observed] - 1]
        output.loc[indexes, FEATURES[2]] = gaps
        # Each outgoing payment enters/leaves the counter once, including ties.
        recipients = group.recipient.to_numpy()
        counts = Counter()
        added = removed = 0
        unique = np.zeros(len(times), dtype=np.int64)
        for i in range(len(times)):
            while added < before[i]:
                counts[recipients[added]] += 1
                added += 1
            while removed < lower24[i]:
                recipient = recipients[removed]
                counts[recipient] -= 1
                if counts[recipient] == 0:
                    del counts[recipient]
                removed += 1
            unique[i] = len(counts)
        output.loc[indexes, FEATURES[3]] = unique
        output.loc[indexes, FEATURES[4]] = _window_sum(times, group.paid_usd.to_numpy(), times)
        received = incoming.get(wallet)
        if received is not None:
            output.loc[indexes, FEATURES[5]] = _window_sum(
                received.timestamp.to_numpy(), received.received_usd.to_numpy(), times)
    return output



## 3. Connect Person 1's fields and value payments

ACCOUNT is the original sender; Destination in EXTRA_TRANSACTION_FIELDS is the final recipient. Paid USD comes from the first executed transfer leg. Received USD comes from DELIVERED_AMOUNT and is checked against the final leg and Person 1's delivered_usd. Prices use the containing UTC hour and issuer/currency identity. We do not sum conversion legs, use SendMax as spending, count transfer legs as transactions or add fees. Missing rates fail clearly. Calendar features remain Person 1's work.


In [ ]:
def prepare_payments(cleaned, transfers, rates):
    """Project one customer payment per cleaned row; ignore labels entirely."""
    required = ['TRANSACTION_HASH', 'TIMESTAMP', 'ACCOUNT',
                'EXTRA_TRANSACTION_FIELDS', 'DELIVERED_AMOUNT']
    missing = set(required) - set(cleaned.columns)
    if missing:
        raise ValueError(f'Missing cleaned fields: {sorted(missing)}')
    payments = cleaned[required].copy().reset_index(drop=True)
    if payments.TRANSACTION_HASH.isna().any() or not payments.TRANSACTION_HASH.is_unique:
        raise ValueError('Cleaned transaction hashes must be nonmissing and unique.')
    payments['FROM_ADDRESS'] = payments.ACCOUNT
    payments['TO_ADDRESS'] = payments.EXTRA_TRANSACTION_FIELDS.map(
        lambda value: json.loads(value)['Destination'])
    # Validate identifiers/timestamps before any rate lookup or integer conversion.
    payments['PAID_USD'] = np.nan
    payments['RECEIVED_USD'] = np.nan
    _validated(payments, Columns())
    payments['TIMESTAMP'] = pd.to_numeric(payments.TIMESTAMP).astype('int64')

    fields = ['TRANSACTION_HASH', 'TRANSFER_INDEX', 'TIMESTAMP', 'FROM_ADDRESS',
              'TO_ADDRESS', 'TOKEN_ADDRESS', 'SYMBOL', 'VALUE']
    if not set(fields).issubset(transfers.columns):
        raise ValueError('Missing required transfer fields.')
    if transfers[fields[:-1]].isna().any().any():
        raise ValueError('Missing transfer identifiers or metadata.')
    if transfers.duplicated(['TRANSACTION_HASH', 'TRANSFER_INDEX']).any():
        raise ValueError('Duplicate executed transfer keys.')
    if set(transfers.TRANSACTION_HASH) != set(payments.TRANSACTION_HASH):
        raise ValueError('Cleaned and transfer transaction hash sets differ.')
    legs = transfers[fields].copy()
    indexes = pd.to_numeric(legs.TRANSFER_INDEX, errors='raise').to_numpy(dtype=float)
    if not np.isfinite(indexes).all() or (indexes < 0).any() or (indexes != np.floor(indexes)).any():
        raise ValueError('Invalid transfer ordering indexes.')
    legs['TRANSFER_INDEX'] = indexes.astype('int64')
    expected_times = legs.TRANSACTION_HASH.map(payments.set_index('TRANSACTION_HASH').TIMESTAMP)
    if not (legs.TIMESTAMP.to_numpy() == expected_times.to_numpy()).all():
        raise ValueError('Transfer and cleaned payment timestamps disagree.')
    ordered = legs.sort_values(['TRANSACTION_HASH', 'TRANSFER_INDEX'])
    first = ordered.drop_duplicates('TRANSACTION_HASH').set_index('TRANSACTION_HASH').reindex(payments.TRANSACTION_HASH)
    last = ordered.drop_duplicates('TRANSACTION_HASH', keep='last').set_index('TRANSACTION_HASH').reindex(payments.TRANSACTION_HASH)
    if not np.array_equal(first.FROM_ADDRESS.to_numpy(), payments.FROM_ADDRESS.to_numpy()):
        raise ValueError('First leg sender differs from original customer sender.')
    if not np.array_equal(last.TO_ADDRESS.to_numpy(), payments.TO_ADDRESS.to_numpy()):
        raise ValueError('Final leg recipient differs from customer destination.')

    rates = rates[['TIMESTAMP', 'ASSET', 'PRICE_MICRO_USD']].copy()
    rates['TIMESTAMP'] = pd.to_numeric(rates.TIMESTAMP, errors='raise')
    rates['PRICE_MICRO_USD'] = pd.to_numeric(rates.PRICE_MICRO_USD, errors='raise')
    if rates.isna().any().any() or rates.duplicated(['TIMESTAMP', 'ASSET']).any():
        raise ValueError('Missing or duplicate reference rates.')
    if (not np.isfinite(rates.TIMESTAMP).all() or (rates.TIMESTAMP % 3600 != 0).any()
            or not np.isfinite(rates.PRICE_MICRO_USD).all() or (rates.PRICE_MICRO_USD <= 0).any()):
        raise ValueError('Invalid hourly reference prices.')
    lookup = rates.set_index(['TIMESTAMP', 'ASSET']).PRICE_MICRO_USD

    def value_usd(assets, amounts):
        keys = pd.MultiIndex.from_arrays([payments.TIMESTAMP // 3600 * 3600, assets])
        prices = lookup.reindex(keys).to_numpy(dtype=float) / 1_000_000
        if np.isnan(prices).any():
            raise ValueError('Missing containing-hour reference price; no stale-rate fallback.')
        amounts = np.asarray(amounts, dtype=float)
        if np.isinf(amounts).any() or (amounts < 0).any():
            raise ValueError('Invalid amount; missing amounts are allowed but never zero-filled.')
        return amounts * prices

    def leg_value(leg):
        assets = leg.TOKEN_ADDRESS + ':' + leg.SYMBOL
        assets = assets.where(leg.TOKEN_ADDRESS != XRP, 'XRP')
        return value_usd(assets, pd.to_numeric(leg.VALUE, errors='raise'))

    # Raw transfer VALUE is asset units (including XRP units), not USD.
    payments['PAID_USD'] = leg_value(first)
    decoded = payments.DELIVERED_AMOUNT.map(json.loads)
    assets = decoded.map(lambda amount: f"{amount['issuer']}:{amount['currency']}" if isinstance(amount, dict) else 'XRP')
    amounts = decoded.map(lambda amount: float(Decimal(amount['value'])) if isinstance(amount, dict) else int(amount) / 1_000_000)
    payments['RECEIVED_USD'] = value_usd(assets, amounts)
    final_usd = leg_value(last)
    known = np.isfinite(final_usd)
    if not np.allclose(final_usd[known], payments.RECEIVED_USD.to_numpy()[known], rtol=1e-9, atol=1e-6):
        raise ValueError('Final leg value and delivered amount disagree.')
    if 'delivered_usd' in cleaned:
        received = pd.to_numeric(cleaned.delivered_usd, errors='raise').to_numpy(dtype=float)
        if not np.allclose(received, payments.RECEIVED_USD, rtol=1e-9, atol=1e-6, equal_nan=True):
            raise ValueError('Person 1 delivered_usd and exact-hour valuation disagree.')
    return payments[['TRANSACTION_HASH', 'TIMESTAMP', 'FROM_ADDRESS', 'TO_ADDRESS', 'PAID_USD', 'RECEIVED_USD']]


## 4. Check boundary behaviour on small examples


In [ ]:
# Small independent reference checks. These run before reading the real data.
def reference_features(data):
    rows = []
    for row in data.itertuples(index=False):
        past = data[data.TIMESTAMP < row.TIMESTAMP]
        window = past[past.TIMESTAMP >= row.TIMESTAMP - 86400]
        outgoing = window[window.FROM_ADDRESS == row.FROM_ADDRESS]
        incoming = window[window.TO_ADDRESS == row.FROM_ADDRESS]
        previous = past[past.FROM_ADDRESS == row.FROM_ADDRESS]
        def total(values):
            return np.nan if values.isna().any() else values.sum()
        rows.append([row.TRANSACTION_HASH,
            int((outgoing.TIMESTAMP >= row.TIMESTAMP - 3600).sum()), len(outgoing),
            np.nan if previous.empty else row.TIMESTAMP - previous.TIMESTAMP.max(),
            outgoing.TO_ADDRESS.nunique(), total(outgoing.PAID_USD), total(incoming.RECEIVED_USD)])
    return pd.DataFrame(rows, columns=['TRANSACTION_HASH'] + FEATURES)

sample = pd.DataFrame([
    ('a', 0, 'A', 'B', 10., 9.), ('b', 0, 'A', 'C', 20., 18.),
    ('c', 1, 'B', 'A', 7., 6.), ('d', 3600, 'A', 'C', 30., 28.),
    ('e', 3600, 'A', 'D', np.nan, 37.), ('f', 86400, 'A', 'B', 50., 48.),
    ('g', 86401, 'A', 'B', 60., 57.), ('h', 172802, 'A', 'B', 70., 65.),
    ('i', 3600, 'Z', 'A', 100., 90.),
], columns=['TRANSACTION_HASH','TIMESTAMP','FROM_ADDRESS','TO_ADDRESS','PAID_USD','RECEIVED_USD'])
sample = sample.sample(frac=1, random_state=7).reset_index(drop=True)
original = sample.copy(deep=True)
pd.testing.assert_frame_equal(build_features(sample), reference_features(sample), check_dtype=False)
pd.testing.assert_frame_equal(sample, original)
assert build_features(sample.iloc[:1]).sender_tx_count_24h.iloc[0] == 0
try:
    build_features(pd.concat([sample, sample.iloc[:1]]))
except ValueError:
    pass
else:
    raise AssertionError('Duplicate transaction was not rejected.')
print('PASS: exact boundaries, simultaneous timestamps, first transactions, separate wallets, missing amounts, unchanged input and duplicate rejection.')


## Generate training features only

Compute the exact six features for the current cleaned training rows. Preserve missing gaps for first observed transactions. Official validation/test are not read. Save and round-trip-verify the training feature table before joining to ML.

In [ ]:
def generate_split(split):
    cleaned_path = CLEANED_DIR / f'{split}_cleaned.parquet'
    raw_dir = RAW_DIR / split
    if not cleaned_path.exists():
        raise FileNotFoundError(f'Run Person 1 cleaning first: missing {cleaned_path}')
    cleaned = pd.read_parquet(cleaned_path)
    transfers = pd.read_parquet(raw_dir / 'transfers.parquet')
    rates = pd.read_csv(raw_dir / 'market_rates.csv')
    payments = prepare_payments(cleaned, transfers, rates)
    features = build_features(payments)
    assert list(features.columns) == ['TRANSACTION_HASH'] + FEATURES
    assert features.TRANSACTION_HASH.is_unique
    assert features.TRANSACTION_HASH.tolist() == cleaned.TRANSACTION_HASH.tolist()
    return features

# Integration experiment: train only; official test is not read.
train_cleaned_features = generate_split('train')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
output = OUTPUT_DIR / 'train_cleaned_features.parquet'
train_cleaned_features.to_parquet(output, index=False)
pd.testing.assert_frame_equal(pd.read_parquet(output), train_cleaned_features)
print(f'train: {len(train_cleaned_features):,} rows, hash + six features, no labels → {output}')
print('Missing values by feature:', train_cleaned_features.isna().sum().to_dict())


## ML handoff

Use `outputs/thanmaya_behaviour/train_cleaned_features.parquet` as the behaviour input in `02_ml_baseline.ipynb`. Preserve legitimate missing gaps and let LightGBM handle them natively. Run the same seed-42 stratified 80/20 split and unchanged original team parameters. Compare AP with our stored baseline. The upstream source's seed-0 recommendation is not used in this comparison.

AI assistance: OpenAI Codex adapted the execution scope and output path for local train-only integration. Thanmaya's original feature definitions and reference tests remain unchanged. No official test generation or submission is performed.